Basic Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Load the dataset

In [2]:
data = pd.read_csv("insurance.csv")

data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'insurance.csv'

In [ ]:
data.shape

In [ ]:
data.info()

Check the feature correlations

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))

corr = data.corr()
sns.heatmap(corr , annot = True , ax=ax)

Save Object

In [ ]:
import pickle

def save_object(obj , name):
    pickle_obj = open(f"{name}.pck","wb")
    pickle.dump(obj, pickle_obj)
    pickle_obj.close()

Label Encode Object Types

In [ ]:
d_types = dict(data.dtypes)
for name , type_ in d_types.items():
    if str(type_) == 'object':
        print(f"<======== {name} ===========>")
        print(data[name].value_counts())
        print()

In [ ]:
from sklearn.preprocessing import LabelEncoder

for name , type_ in d_types.items():
    if str(type_) == 'object':
        Le = LabelEncoder()
        data[name] = Le.fit_transform(data[name])
        save_object(Le , f"Label_Encoder_{name}")

Check info after Label Encoding

In [ ]:
data.info()

One hot Encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder

onehotencoder = OneHotEncoder()
part = onehotencoder.fit_transform(data['region'].values.reshape(-1,1)).toarray()
save_object(onehotencoder , "OneHotEncoder_region")

values = dict(data["region"].value_counts())

for e , (val , _) in enumerate(values.items()):
    data["region_" + str(val)] = part[:,e]

data = data.drop(["region"] , axis = 1)

data.head()

In [ ]:
data.info()

Handle Skewness in Predictive column

In [ ]:
Original_Y = data["expenses"].values.copy()

In [ ]:
Original_Y

In [ ]:
print("Skewness in Column : Expenses " , data["expenses"].skew())

plt.hist(data["expenses"])
plt.show()

In [ ]:
col_log = np.log(data["expenses"])
print("Skewness in Column : Log Expenses " , col_log.skew())

plt.hist(col_log)
plt.show()

In [ ]:
col_sqrt = np.sqrt(data["expenses"])

print("Skewness in Column : Sqrt Expenses " ,col_sqrt.skew())

plt.hist(col_sqrt)
plt.show()

In [ ]:
from scipy import stats

col_cox , lam = stats.boxcox(data["expenses"])[0:2]
print("Skewness in Column : Sqrt Expenses " ,pd.Series(col_cox).skew())

save_object(lam , "boxcox_lambda")

plt.hist(col_cox)
plt.show()

In [ ]:
data["expenses"] = col_cox

Make Features and Targets

In [ ]:
remaining_columns = list(data.columns)
remaining_columns.remove("expenses")

In [ ]:
save_object(remaining_columns , "columns")

In [ ]:
X = data[remaining_columns].values
Y = data['expenses'].values

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Xtrain , Xtest , Ytrain , Ytest = train_test_split(X , Y , test_size = 0.2 , random_state = 4)

Scaler = StandardScaler()
Xtrain = Scaler.fit_transform(Xtrain)
Xtest = Scaler.transform(Xtest)

save_object(Scaler , "Scaler")

In [ ]:
# check whether data is standardized or not
# mean should be 1

plt.ylim(-1,1)

means = []
for i in range(Xtrain.shape[1]):
    means.append(np.mean(Xtrain[:,i]))
plt.plot(means , scaley=False)

In [ ]:
# Check variances

plt.ylim(0,2)

vars = []
for i in range(Xtrain.shape[1]):
    vars.append(np.var(Xtrain[:,i]))
plt.plot(vars)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA

pca = PCA(n_components = 7)
Xtrain = pca.fit_transform(Xtrain)
Xtest = pca.transform(Xtest)

pca.explained_variance_ratio_.cumsum()

save_object(pca , "PCA")

Defining Metrics

In [ ]:
def rmse_score(y_test , y_pred):
    value = (1/len(y_test))*np.sum((y_test - y_pred)**2)
    return np.sqrt(value)

def r2_score(y_test , y_pred):
    numenator = (1/len(y_test))*np.sum((y_test - y_pred)**2)
    denominator = (1/len(y_test))*np.sum((y_test - np.mean(y_test))**2)
    return (1 - (numenator/denominator))

def mae(y_test , y_pred):
    return (1/len(y_test))*np.sum(np.abs(y_test - y_pred))

def adj_r2_score(y_test , y_pred , n_features):
    numenator = (1-r2_score(y_test , y_pred))*(len(y_test) - 1)
    denominator = len(y_test) - n_features - 1
    return 1 - (numenator/denominator)

In [ ]:
model = LinearRegression()
model.fit(Xtrain , Ytrain)

Ypred = model.predict(Xtest)

print("rmse_score : " , rmse_score(Ytest , Ypred))
print("r2_score : " , r2_score(Ytest , Ypred))
print("mae : " , mae(Ytest , Ypred))
print("adj_r2_score : " , adj_r2_score(Ytest , Ypred , Xtest.shape[1]))

In [ ]:
save_object(model , "MyModel")

Realtime Prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

In [ ]:
# Values
# Label Encoding
# OneHotEncoding
# Scaler
# PCA
# Predict
# Inverse Boxcox

In [ ]:
def load_object(name):
    pickle_obj = open(f"{name}.pck","rb")
    obj = pickle.load(pickle_obj)
    return obj

In [ ]:
# Load the Data Point

data = pd.read_csv("insurance.csv")

idx = np.random.choice(len(data))
to_be_predicted = data.iloc[idx,:].values

col_names = data.columns
predict_dict = {}

for col_name , val in zip(col_names , to_be_predicted):
    predict_dict[col_name] = val

print(predict_dict)

In [ ]:
real_value = predict_dict["expenses"]
del predict_dict["expenses"]

In [ ]:
predict_dict["region"] = load_object("Label_Encoder_region").transform(np.array(predict_dict["region"]).reshape(-1,))

In [ ]:
predict_dict["sex"] = load_object("Label_Encoder_sex").transform(np.array(predict_dict["sex"]).reshape(-1,))[0]

In [ ]:
predict_dict["smoker"] = load_object("Label_Encoder_smoker").transform(np.array(predict_dict["smoker"]).reshape(-1,))[0]

In [ ]:
predict_dict

In [ ]:
predict_dict["region_ohe"] = load_object("OneHotEncoder_region").transform(predict_dict["region"].reshape(-1,1)).toarray()[0]

In [ ]:
predict_dict

In [ ]:
del predict_dict["region"]

In [ ]:
for e , i in enumerate(predict_dict["region_ohe"]):
    predict_dict["region_" + str(e)] = i

In [ ]:
del predict_dict["region_ohe"]

In [ ]:
predict_dict

In [ ]:
# Lets make the main array

col_sequence = load_object("columns")
array = []

for col_name in col_sequence :
    array.append(predict_dict[col_name])

array = np.array(array)

print(array)

In [ ]:
array = load_object("Scaler").transform(array.reshape(1,-1))

In [ ]:
array = load_object("PCA").transform(array)

In [ ]:
array

In [ ]:
prediction = load_object("MyModel").predict(array)
print(prediction)

In [ ]:
from scipy.special import inv_boxcox

prediction = inv_boxcox(prediction , load_object("boxcox_lambda"))

print(prediction)

In [ ]:
print(" Original " , real_value , " , Predicted " , float(prediction[0]))